<a href="https://colab.research.google.com/github/dgylayse/AkademiQ_DataScience/blob/main/AkademiQ_Data_Science_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Veri Dünyası

**Etiket varsa -> Denetimli Öğrenme**
- Kategorik çıktı -> Sınıflandırma
- Sürekli çıktı -> Regresyon

**Etiket yoksa -> Denetimsiz Öğrenme**
- Segment bulma -> Kümeleme
- Yapı sıkıştırma -> Boyut indirgeme

**Ortalama etkileşim varsa -> Takviyeli Öğrenme**
- Aksiyon
- Ödül
- Politika güncelleme



In [2]:
import re #metinlerde desen arama ve temizleme
import time #kod çalışma süresini hesaplar
import warnings #uyarı mesajlarını gizlemek ve yönetmek için
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.compose import ColumnTransformer #sayısal ve kategori sütunlara farklı işlemleri uygulamak için
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer #eksik verileri doldurmak için
from sklearn.preprocessing import OrdinalEncoder #kategorik verileri sayısala çevirmek için
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [9]:
df = pd.read_csv("Food_Delivery_Times.csv")
df.head()

,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
0,522,7.93,Windy,Low,Afternoon,Scooter,12,1.0,43
1,738,16.42,Clear,Medium,Evening,Bike,20,2.0,84
2,741,9.52,Foggy,Low,Night,Scooter,28,1.0,59
3,661,7.44,Rainy,Medium,Afternoon,Scooter,5,1.0,37
4,412,19.03,Clear,Low,Morning,Bike,16,5.0,68


In [32]:
def clean_col(c):
  c = c.lower().strip()
  c = re.sub(r"[^a-z0-9]+","_", c) #Harf ve sayı haricindeki tüm karakterleri "_" ile değiştirir
  return c.strip("_")
df.columns = [clean_col(c) for c in df.columns] # Sütunların isimlerini temizledik

print("\nSÜTUNLAR:",df.columns.tolist())

#target_candidates = [c for c in df.columns if "time" in c]

#print("\nTARGET:",target_candidates)

#TARGET = target_candidates[0]


SÜTUNLAR: ['distance_km', 'weather', 'traffic_level', 'time_of_day', 'vehicle_type', 'preparation_time_min', 'courier_experience_yrs', 'delivery_time_min']


In [31]:
#df[TARGET]=(
 #   df[TARGET].astype(str).str.extract(r"(\d+\.?\d*)")[0].astype(float))

#df = df.dropna(subset=[TARGET]) #NaN olan ifadeleri veri setinden kaldırır.
#print(df[TARGET].head())

#30 mins -> 30
#12.569 minutes -> 12.5
#450 min -> 450

In [34]:
TARGET = "delivery_time_min"
print("Target:", TARGET)

df[TARGET]=(
    df[TARGET].astype(str).str.extract(r"(\d+\.?\d*)")[0].astype(float))

df = df.dropna(subset=[TARGET]).reset_index(drop=True)
print("Target ilk değerler:")
display(df[TARGET].head())

Target: delivery_time_min
Target ilk değerler:


,delivery_time_min


In [35]:
drop_like_id = [c for c in df.columns if c in {"id","person_id"} or c.endswith("_id")]
df = df.drop(columns=drop_like_id, errors = "ignore")

print("ID sonrası şekil:", df.shape)

ID sonrası şekil: (0, 8)


In [36]:
# Eğitim ve test seti ayırma

X = df.drop(columns=[TARGET]).copy() # algoritmaya girdi
y = df[TARGET].copy() #algoritmanın hedefi


In [37]:
time_col = next((c for c in X.columns if "order" in c and "time" in c), None)

if time_col:
  dt = pd.to_datetime(X[time_col], fotmat="%H:%M:%S", errors = "coerce") #errors=coerce ifadesi hatalı bir saat varsa bozma, boş(NaT) yap
  X["order_hour"] = dt.dt.hour #10:18:24 formatındaki zamansal veriden sadece saat kısmını alıp yeni sütun olarak kaydetmesi için
  X = X.drop(columns=[time_col])


In [40]:
lat_cols = [c for c in X.columns if "lat" in c]
lon_cols = [c for c in X.columns if "long" in c or "lng" in c]

if len(lat_cols) >= 2 and len(lon_cols) >=2:
  r_lat, d_lat = lat_cols[:2] #[restaurant_latitude,delivery_location_latitude]
  r_lon, d_lon = lon_cols[:2]
  X["geo l1 distance"] = (X[r_lat] - X[d_lat]).abs() + (X[r_lon]- X[d_lon]).abs()


In [41]:
if "weather" in X.columns:
  X["bad_weather"] = X["weather"].astype(str).str.lower().isin(
      ["rainy","stormy","foggy","snowy"]
  ).astype(int)

In [42]:
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print("Kategorik sütunlar:", cat_cols)
print("Sayısal sütunlar:", len(num_cols))



Kategorik sütunlar: ['weather', 'traffic_level', 'vehicle_type']
Sayısal sütunlar: 5


#

In [45]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(max_depth=8, random_state=42),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
}
